# 09 — Síntese comparativa

Integra somente artefatos anteriores e exporta tabelas, HTML e figuras finais.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
REPO_DIR = Path("/content/falando_nela")
REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_REF = ""  # Opcional: branch, tag ou commit; vazio acompanha o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-analise.txt"], check=True)
print("Data root:", DATA_ROOT)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())

## Configuração

Use o mesmo `RUN_ID` em toda a suíte. A configuração versionada é a fonte de verdade.

In [ ]:
from analise.discursos_plenario.config import load_config, resolve_input_paths, resolve_output_root

RUN_ID = "analise-plenario-20260713-v1"
CONFIG_PATH = REPO_DIR / "analise" / "discursos_plenario" / "config.v1.json"
ANALYSIS_CONFIG = load_config(CONFIG_PATH)
RUN_OUTPUT_ROOT = resolve_output_root(ANALYSIS_CONFIG, DATA_ROOT, RUN_ID)
INPUT_PATHS = resolve_input_paths(ANALYSIS_CONFIG, DATA_ROOT)
RODAR_ETAPA = False

assert ANALYSIS_CONFIG.date_start == "2010-02-02"
assert ANALYSIS_CONFIG.date_end == "2026-07-13"
assert ANALYSIS_CONFIG.raw["complete_year_end"] == 2025
assert ANALYSIS_CONFIG.raw["ytd_year"] == 2026
print("Run:", RUN_ID)
print("Saida:", RUN_OUTPUT_ROOT)

## Decisão metodológica

Resultados permanecem separados por arena; comparações padronizadas são secundárias. Reprodução, robustez e exploração aparecem identificadas.

In [ ]:
SINTESE_SNAPSHOT_PATH = RUN_OUTPUT_ROOT / "00_snapshot" / "discursos_plenario_snapshot.parquet"
assert SINTESE_SNAPSHOT_PATH.exists(), "Execute o caderno 00."
SINTESE_MANIFESTS = sorted(RUN_OUTPUT_ROOT.glob("*/manifest*.json"))
print("Manifests disponíveis:", len(SINTESE_MANIFESTS))
for SINTESE_MANIFEST_PATH in SINTESE_MANIFESTS:
    print(SINTESE_MANIFEST_PATH.relative_to(RUN_OUTPUT_ROOT))

## Execução

A etapa cara permanece desativada até a inspeção das entradas e dos parâmetros acima.

In [ ]:
from analise.discursos_plenario.sintese import run_synthesis

SINTESE_RESULT = None
if RODAR_ETAPA:
    SINTESE_RESULT = run_synthesis(data_root=DATA_ROOT, run_id=RUN_ID, config_path=CONFIG_PATH)
    print(SINTESE_RESULT["manifest_path"])
else:
    print("Síntese não executada.")

## Validação imediata

Esta checagem não substitui os testes sintéticos nem a revisão dos manifests.

In [ ]:
import pandas as pd

SINTESE_COVERAGE_PATH = RUN_OUTPUT_ROOT / "09_sintese" / "cobertura.csv"
if SINTESE_COVERAGE_PATH.exists():
    SINTESE_COVERAGE = pd.read_csv(SINTESE_COVERAGE_PATH)
    assert SINTESE_COVERAGE.loc[SINTESE_COVERAGE["ano"].eq(2026), "ytd"].all()
    SINTESE_EXPECTED = ["cobertura.parquet", "sintese.html", "discursos_por_arena.svg", "discursos_por_arena.png"]
    assert all((RUN_OUTPUT_ROOT / "09_sintese" / name).exists() for name in SINTESE_EXPECTED)
    display(SINTESE_COVERAGE.tail(12))